## Compañia OilyGiant 

La compañia es de extracción de petroleo y la tarea principal es encontrar los mejores lugares donde **abrir 200 pozos nuevos de petroleo minimizando los riesgos y maximizando los benefecios**


### 1. Carga y preparación de los datos

En la carga de archivos se leyeron los tres datasets (geo_data_0.csv, geo_data_1.csv, geo_data_2.csv) que contienen información de pozos de petróleo. 
- Cada dataset cuenta con 100 000 registros y 5 columnas:
  - id: identificador único del pozo (no se usa en el modelado).
  - f0, f1, f2: tres características numéricas de los pozos (de acuerdo a la información suministrada inicialmente para el proyecto su significado especifico no es importante, pero las características en sí son significativas).
  - **product: volumen de reservas en el pozo de petróleo (en miles de barriles), es la variable objetivo.**


- Se verificó el tipo de datos: f0, f1, f2 y product son float64, id es object.
- No se encontraron valores nulos ni duplicados en ninguna región.

Análisis descriptivo:
- Región 0:
  - Promedio de reservas: 92.5 mil barriles.
  - Distribución relativamente simétrica, rango entre 0 y 185 mil.

- Región 1:
  - Promedio de reservas: 68.8 mil barriles.
  - Mucha dispersión en las features (f0 y f1 tienen valores muy grandes en positivo y negativo).
  - Reservas con rango entre 0 y 137 mil.

- Región 2:
  - Promedio de reservas: 95 mil barriles.
  - Distribución estable, reservas entre 0 y 190 mil.


- Los datos son limpios y no requieren imputación ni depuración inicial.
- La variable id se descarta para el modelado.
- f0, f1 y f2 se mantienen como características.
- El product es la variable objetivo a predecir.
- A simple vista, la región 2 parece tener un promedio de reservas ligeramente superior a las demás, pero esto debe confirmarse con el modelo y el análisis de beneficios.

In [17]:
#Importar las librerias necesarias
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [18]:
# Cargar los tres datasets
data0 = pd.read_csv('datasets/geo_data_0.csv')
data1 = pd.read_csv('datasets/geo_data_1.csv')
data2 = pd.read_csv('datasets/geo_data_2.csv')

# Inspección básica
for i, d in enumerate([data0, data1, data2]):
    print(f"--- Región {i} ---")
    print(d.info())
    print(d.head())
    print(d.describe())
    print("Valores nulos:", d.isna().sum().sum())
    print("Duplicados:", d.duplicated().sum())
    print()

--- Región 0 ---
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  str    
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), str(1)
memory usage: 3.8 MB
None
      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
                  f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433       3.248248     

### 2. Entrenar y probar el modelo para cada region en *geo_data_0.csv*

In [20]:
# separar features y target
features0 = data0.drop(['id', 'product'], axis=1)
target0 = data0['product']

# 2.1 dividir en train y valid (75:25)
features_train, features_valid, target_train, target_valid = train_test_split(
    features0, target0, test_size=0.25, random_state=12345
)

# 2.2 entrenar modelo
model = LinearRegression()
model.fit(features_train, target_train)

# 2.3 predicciones
predictions = model.predict(features_valid)

# 2.4 calcular métricas
rmse = root_mean_squared_error(target_valid, predictions)
mean_pred = predictions.mean()

print("Volumen medio de reservas predicho:", mean_pred)
print("RMSE del modelo:", rmse)

Volumen medio de reservas predicho: 92.59256778438038
RMSE del modelo: 37.5794217150813


**Resultados de la región 0 (geo_data_0.csv)**
- Volumen medio de reservas predicho: 92.6 mil barriles. 
    Esto está muy alineado con la media real de la región (92.5 mil barriles). Indica que el modelo no tiene un sesgo sistemático: acierta bastante bien en promedio.
  
- Error cuadrático medio (RMSE): 37.6 mil barriles.
    El error es significativo: si lo comparamos con la media (92.5), el modelo puede equivocarse en más de un tercio del valor esperado, esto confirma que la regresión lineal no predice con gran precisión cada pozo de manera individual.

In [21]:
# 2.6 se realiza y ejecuta los pasos 2.1-2.5 para los archivos 
# 'geo_data_1.csv' y 'geo_data_2.csv'.

def train_and_validate(path, region_name):
    data = pd.read_csv(path)
    features = data.drop(['id', 'product'], axis=1)
    target = data['product']
    
    # dividir
    X_train, X_valid, y_train, y_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345
    )
    
    # entrenar
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # predicciones
    predictions = model.predict(X_valid)
    
    # métricas
    rmse = mean_squared_error(y_valid, predictions)
    mean_pred = predictions.mean()
    
    print(f"--- Resultados {region_name} ---")
    print("Volumen medio de reservas predicho:", mean_pred)
    print("RMSE:", rmse)
    print()
    
    return predictions, y_valid, model

# ejecutar para las tres regiones
pred0, y0, model0 = train_and_validate('datasets/geo_data_0.csv', "Región 0")
pred1, y1, model1 = train_and_validate('datasets/geo_data_1.csv', "Región 1")
pred2, y2, model2 = train_and_validate('datasets/geo_data_2.csv', "Región 2")


--- Resultados Región 0 ---
Volumen medio de reservas predicho: 92.59256778438038
RMSE: 1412.2129364399243

--- Resultados Región 1 ---
Volumen medio de reservas predicho: 68.72854689544603
RMSE: 0.7976263360391149

--- Resultados Región 2 ---
Volumen medio de reservas predicho: 94.96504596800492
RMSE: 1602.377581323619



Región 0: El modelo predice una media muy cercana a la real (92.5), pero el error es alto (37.6). Esto significa que las predicciones individuales varían mucho, aunque la media esté bien ajustada.

Región 1: Llama la atención el RMSE extremadamente bajo (0.89). Esto indica que el modelo prácticamente predice perfecto en validación. Sin embargo, hay que tener cuidado, los datos están generados de forma que en esta región la variable objetivo depende casi de manera exacta de una de las features, por eso la regresión lineal ajusta casi sin error.
El problema es que la media de reservas (68.7) es más baja que en las otras regiones.

Región 2: Media predicha muy alta (95.0), incluso superior a la región 0. El error (40.0) también es considerable, parecido al de la región 0, lo que muestra dispersión en las predicciones.

### 3. cálculo de ganancias

#### 3.1 Variables necesarias
Según las condiciones:
- Presupuesto total: 100 millones USD
- Número de pozos a desarrollar: 200
- Precio por 1k barriles: 4.5 USD por barril

In [22]:
BUDGET = 100_000_000      # presupuesto total en dólares
OIL_PRICE = 4.5           # USD por barril
UNITS_PRICE = 4500        # USD por cada unidad (mil barriles)
NUMBER_OF_WELLS = 200     # pozos a desarrollar

#### 3.2 Umbral de rentabilidad

El presupuesto se reparte entre 200 pozos, cada pozo debe cubrir 500,000 USD en ingresos para no generar pérdidas.

En términos de reservas: 500,000 / 4500 = 111.1 Es decir, cada pozo debe tener al menos 111.1 mil barriles de reservas.

Comparación con los resultados anteriores:
- Región 0: media = 92.6 → por debajo del umbral.
- Región 1: media = 68.7 → muy por debajo del umbral.
- Región 2: media = 95.0 → también por debajo del umbral.

Ninguna región alcanza el umbral de rentabilidad de forma promedio, aun así, el proyecto no se decide con la media: se seleccionarán los 200 pozos con mayores reservas predichas, no un pozo promedio.
Esto significa que, aunque la media esté por debajo de 111.1, puede que la selección de los mejores 200 pozos sí supere el umbral y produzca beneficios.

### 4. Funcion para calcular la ganancia y selección de los 200 mejores pozos por región

4.1 y 4.2
- Se ordenan las predicciones de validación en orden descendente
- Se toman los 200 primeros pozos
- Con esos índices, se recupera el volumen real (target_valid) de reservas
- Se calcula la ganancia teniendo en cuenta que:

  *Ganancia = Suma de reservas reales(Top 200) x 4500 - 100 000 0000*

In [23]:
def calculate_profit(predictions, target, count=200, price=4500, budget=100_000_000):
    # convertir a DataFrame para unir predicciones con valores reales
    df = pd.DataFrame({'predictions': predictions, 'target': target})
    
    # ordenar por predicciones descendente
    top_wells = df.sort_values(by='predictions', ascending=False).head(count)
    
    # volumen real de los pozos seleccionados
    total_product = top_wells['target'].sum()
    
    # ingresos
    revenue = total_product * price
    
    # ganancia neta
    profit = revenue - budget
    
    return profit, top_wells['target'].mean(), total_product

#### 4.3 Aplicar a las tres regiones

In [24]:
profit0, mean0, total0 = calculate_profit(pred0, y0)
profit1, mean1, total1 = calculate_profit(pred1, y1)
profit2, mean2, total2 = calculate_profit(pred2, y2)

print("--- Región 0 ---")
print("Ganancia potencial:", profit0)
print("Volumen total (top 200):", total0)
print("Volumen medio (top 200):", mean0)
print()

print("--- Región 1 ---")
print("Ganancia potencial:", profit1)
print("Volumen total (top 200):", total1)
print("Volumen medio (top 200):", mean1)
print()

print("--- Región 2 ---")
print("Ganancia potencial:", profit2)
print("Volumen total (top 200):", total2)
print("Volumen medio (top 200):", mean2)


--- Región 0 ---
Ganancia potencial: 33208260.43139851
Volumen total (top 200): 29601.83565142189
Volumen medio (top 200): 148.00917825710945

--- Región 1 ---
Ganancia potencial: 24150866.966815114
Volumen total (top 200): 27589.081548181137
Volumen medio (top 200): 137.9454077409057

--- Región 2 ---
Ganancia potencial: 27103499.635998324
Volumen total (top 200): 28245.22214133296
Volumen medio (top 200): 141.2261107066648


#### La Región 0 se posiciona como la más prometedora, ya que sus 200 mejores pozos proyectan la mayor ganancia potencial y un volumen medio de reservas más alto que el de las demás regiones.

La Región 1, a pesar de tener un modelo muy preciso, no alcanza el umbral de rentabilidad deseado debido a su bajo promedio de reservas.

La Región 2 tiene un comportamiento intermedio, pero no logra superar a la Región 0 en términos de ganancias potenciales.

### 5. Análisis de riesgos y ganancias con Bootstrapping

In [25]:
# 5. 1 Parámetros del negocio y del muestreo
BUDGET = 100_000_000            # USD
PRICE_PER_UNIT = 4500           # 4.5 USD/barril * 1000 barriles
SAMPLE_SIZE = 500               # puntos explorados por región
TOP_K = 200                     # pozos a desarrollar
N_BOOT = 1000                   # repeticiones bootstrap
state = np.random.RandomState(12345)

def _profit_from_sample(preds, y_true, idx):
    """
    Dada una muestra de índices (idx):
      1) ordena por predicción desc
      2) toma TOP_K
      3) suma el product real
      4) calcula ganancia = product_sum*PRICE_PER_UNIT - BUDGET
    """
    preds = np.asarray(preds)
    y_true = np.asarray(y_true)

    order = np.argsort(preds[idx])[::-1][:TOP_K]
    chosen = idx[order]
    total_product = y_true[chosen].sum()
    return total_product * PRICE_PER_UNIT - BUDGET

def bootstrap_profits(preds, y_true, n_boot=N_BOOT, sample_size=SAMPLE_SIZE, rng=state):
    """Devuelve una Serie con N_BOOT ganancias simuladas para una región."""
    n = len(preds)
    profits = np.empty(n_boot, dtype=float)
    for b in range(n_boot):
        idx = rng.choice(n, size=sample_size, replace=True)
        profits[b] = _profit_from_sample(preds, y_true, idx)
    return pd.Series(profits, name="profit")

# Generar la distribución de beneficios por región
profits0 = bootstrap_profits(pred0, y0)
profits1 = bootstrap_profits(pred1, y1)
profits2 = bootstrap_profits(pred2, y2)

# Comprobación rápida (para mirar que existen y tienen 1000 muestras)
print(profits0.head(), "\n")
print(profits1.head(), "\n")
print(profits2.head())

0    6.054641e+06
1    5.363934e+06
2    2.937858e+06
3    1.789934e+06
4    2.719929e+06
Name: profit, dtype: float64 

0    3.178949e+06
1    1.010041e+06
2    1.295538e+06
3    5.105900e+06
4    5.512065e+06
Name: profit, dtype: float64 

0    7.433808e+06
1    5.687995e+06
2    2.468884e+06
3    3.901355e+06
4    5.309689e+06
Name: profit, dtype: float64


Cada lista (profits0, profits1, profits2) contiene 1000 valores de ganancia simulados para cada región.

En cada simulación se hizo lo siguiente:
- Se tomó una muestra de 500 pozos con reemplazo.
- Se seleccionaron los 200 pozos con mayores reservas predichas.
- Se calcularon los ingresos usando el valor real (product) de esos 200 pozos.
- Se restó el presupuesto fijo de 100 millones USD.

En los resultados se puede observar que:
- Para la Región 0, las primeras simulaciones muestran ganancias como 6.05M, 5.36M, 2.94M… (en dólares).
- Para la Región 1, los valores son menores, por ejemplo 3.17M, 1.01M, 1.29M…
- Para la Región 2, hay simulaciones con ganancias más altas, como 7.43M, 5.68M, 2.47M…

In [26]:
# 5.2 Beneficio promedio

def summarize_bootstrap(profits, region_name):
    mean_profit = profits.mean()
    ci_low, ci_high = profits.quantile([0.025, 0.975])
    risk = (profits < 0).mean() * 100
    
    print(f"--- {region_name} ---")
    print(f"Ganancia promedio: {mean_profit:,.0f} USD")
    print(f"IC 95%: [{ci_low:,.0f} ; {ci_high:,.0f}] USD")
    print(f"Riesgo de pérdidas: {risk:.2f}%")
    print()
    
    return mean_profit, ci_low, ci_high, risk

res0 = summarize_bootstrap(profits0, "Región 0")
res1 = summarize_bootstrap(profits1, "Región 1")
res2 = summarize_bootstrap(profits2, "Región 2")

--- Región 0 ---
Ganancia promedio: 3,961,650 USD
IC 95%: [-1,112,155 ; 9,097,669] USD
Riesgo de pérdidas: 6.90%

--- Región 1 ---
Ganancia promedio: 4,611,558 USD
IC 95%: [780,508 ; 8,629,521] USD
Riesgo de pérdidas: 0.70%

--- Región 2 ---
Ganancia promedio: 3,929,505 USD
IC 95%: [-1,122,276 ; 9,345,629] USD
Riesgo de pérdidas: 6.50%



#### 5.3 De acuerdo con los criterios de negocio (ganancia promedio alta y riesgo de pérdidas < 2.5%), la única región que cumple los requisitos es la Región 1.

Coincide con que su modelo tiene el menor error (RMSE ≈ 0.89) y predice con mucha precisión.

A pesar de que en el paso 4 la Región 0 parecía más atractiva por la ganancia puntual más alta, al evaluar riesgos se ve que su volatilidad es demasiado elevada.

Por lo tanto, la recomendación final es desarrollar los pozos en la Región 1, ya que combina seguridad (riesgo muy bajo) con un beneficio promedio superior.

### Conclusión final del proyecto

Tras aplicar el modelo de regresión lineal y evaluar los resultados con la técnica de bootstrapping, se pueden concluir:

- Región 0: Aunque presentó una ganancia potencial atractiva en el análisis inicial, el bootstrapping mostró un riesgo de pérdidas del 6.9%, muy superior al umbral máximo aceptado del 2.5%. Esto hace que la región no sea recomendable para la inversión, ya que la volatilidad de las ganancias es demasiado alta.

- Región 1: Destaca por ser la más consistente. La ganancia promedio fue de 4.61 millones de USD, el intervalo de confianza del 95% se mantuvo en valores positivos ([0.78 M ; 8.62 M] USD), y el riesgo de pérdidas fue apenas del 0.7%, muy por debajo del límite permitido. Esto confirma que, aunque la media de reservas en esta región es menor que en otras, la alta precisión del modelo garantiza una selección eficiente de los pozos más rentables.

- Región 2: Al igual que la Región 0, mostró un riesgo de pérdidas elevado (6.5%) y, por tanto, tampoco cumple con las condiciones de negocio para su desarrollo, a pesar de contar con un beneficio promedio similar a la Región 0.